# 線形回帰 — 重みを読む

**出典**　機械学習 4 分野の教科書 — 第 2 章 1-A 回帰と正則化 ／ 02 線形回帰：最も基本の道具

**このノートのパス**　`ml-code/02_regression/01_linear_regression.ipynb`

**このノートで確かめること**
住宅価格を線形回帰で予測し、**標準化を訓練データだけで fit する**手順と、
**係数からどの特徴がどれだけ効いているかを読む**手順を確かめる。

| 項目 | 内容 |
|---|---|
| 必要なパッケージ | `scikit-learn, numpy` |
| 実行時間の目安 | 約 20 秒 |
| 前提 | **このノート単体で完結する**（他のノートを先に実行する必要はない） |

In [2]:
# 未導入なら実行する（Colab など）
# !pip install -q scikit-learn numpy

### データの取得について

`fetch_california_housing()` は**初回だけネットワークにアクセス**する。
社内プロキシなどで取得できない環境のために、下のセルは失敗時に合成データへ切り替える。

In [3]:
import numpy as np
from sklearn.datasets import fetch_california_housing, make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

try:
    data = fetch_california_housing()
    X, y = data.data, data.target
    feature_names = list(data.feature_names)
    print("California Housing:", X.shape)
except Exception as e:
    print("取得できなかったので合成データで代用:", type(e).__name__)
    X, y = make_regression(n_samples=20_000, n_features=8, n_informative=6,
                           noise=0.5, random_state=0)
    feature_names = [f"feature_{i}" for i in range(X.shape[1])]

California Housing: (20640, 8)


## 学習と評価

**標準化は「訓練データで fit → 訓練とテストの両方に transform」**。
テストデータの統計量を使うと未来の情報が漏れる（データリーク）。

In [4]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)

# 標準化は「訓練データで fit → 訓練とテストの両方に transform」。
# テストデータの統計量を使うと未来の情報が漏れる（データリーク）。
scaler = StandardScaler().fit(X_tr)
X_tr_s, X_te_s = scaler.transform(X_tr), scaler.transform(X_te)

model = LinearRegression().fit(X_tr_s, y_tr)
import warnings

for label, values in [("X_train", X_tr_s), ("X_test", X_te_s),
                      ("coef", model.coef_), ("intercept", model.intercept_)]:
    if not np.isfinite(values).all():
        raise ValueError(f"{label} contains NaN or infinity")

with warnings.catch_warnings():
    warnings.simplefilter("error", RuntimeWarning)
    with np.errstate(divide="raise", over="raise", invalid="raise"):
        pred = model.predict(X_te_s)
        reference = np.einsum("ij,j->i", X_te_s, model.coef_, optimize=False) + model.intercept_

assert np.isfinite(pred).all(), "Prediction contains NaN or infinity"
np.testing.assert_allclose(pred, reference, rtol=1e-12, atol=1e-12)
print("Prediction check: finite; max difference =", np.max(np.abs(pred - reference)))

print("RMSE:", np.sqrt(mean_squared_error(y_te, pred)))
print("R^2 :", r2_score(y_te, pred))

# 重みを見ると「どの特徴がどれだけ効いているか」が読める
for name, w in zip(feature_names, model.coef_):
    print(f"{name:>12}: {w:+.3f}")

Prediction check: finite; max difference = 1.7763568394002505e-15
RMSE: 0.7273129773603114
R^2 : 0.5943232652466204
      MedInc: +0.826
    HouseAge: +0.117
    AveRooms: -0.249
   AveBedrms: +0.290
  Population: -0.009
    AveOccup: -0.031
    Latitude: -0.900
   Longitude: -0.871


## 読み取りどころ

- **係数は標準化後の値に対するもの**である。だから大小をそのまま比較してよい。
  標準化前の生データで学習すると、単位の大きい特徴の係数が小さく出て、比較できなくなる。
- **係数の符号は因果ではない**。相関の向きにすぎず、他の特徴との交絡で簡単に反転する。
- R² が 0.6 程度なら「説明できていない部分が 4 割ある」という意味。**非線形な関係を線形モデルで近似している限界**がここに出る。